# Claude API — Request Lifecycle

---

## Overview

Every interaction with Claude follows **five distinct phases**:
```
User (Client)
    |
    | 1. Request to your server
    v
Your Server
    |
    | 2. Request to Anthropic API  (with secret API key)
    v
Anthropic API
    |
    | 3. Model processing
    v
Anthropic API
    |
    | 4. Response to your server
    v
Your Server
    |
    | 5. Response to client
    v
User (Client)
```

---

## Why You Need a Server (Never Call the API from the Client)

| Concern | Detail |
|---|---|
| **Security** | API key must never be exposed in client-side code |
| **Unauthorized use** | Anyone can extract a key from browser/app code |
| **Solution** | Client talks to your server; your server talks to Anthropic |

---

## Making API Requests

Anthropic provides official SDKs for: **Python, TypeScript, JavaScript, Go, Ruby**  
You can also use plain HTTP requests.

### Required Fields in Every Request

| Field | Purpose |
|---|---|
| `api_key` | Authenticates your request to Anthropic |
| `model` | Which Claude model to use (e.g. `claude-3-sonnet`) |
| `messages` | List containing the user's input |
| `max_tokens` | Upper limit on tokens Claude can generate |

---

## Inside Claude's Processing (4 Stages)

### 1. Tokenization
- Input text is broken into **tokens** — words, subwords, spaces, or symbols
- Rule of thumb: ~1 word ≈ 1 token

### 2. Embedding
- Each token is converted into an **embedding** — a long list of numbers encoding all
  possible meanings of that token
- Captures semantic relationships between words

**Example — "quantum" could mean:**
- A discrete unit of physical quantity (physics)
- Quantum mechanics / quantum physics
- Something subatomic or extremely small
- Quantum computing

### 3. Contextualization
- Embeddings are **refined based on surrounding words** to determine the most likely meaning
- Self-attention adjusts numerical representations to highlight the appropriate definition
- This is where the Transformer architecture does its core work

### 4. Generation
- Contextualized embeddings pass through an output layer that calculates **probability
  distributions** over all possible next tokens
- Claude does **not** always pick the highest probability token — it uses controlled
  randomness to produce natural, varied responses
- After each token is selected, it is added to the sequence and the process repeats
```
[Contextualized Embeddings]
         |
         v
[Output Layer — probability over vocab]
         |
         v
[Sample next token]
         |
         v
[Append to sequence → repeat]
```

---

## When Claude Stops Generating

After each token, Claude checks three stopping conditions:

| Condition | Description |
|---|---|
| **Max tokens reached** | Hit the `max_tokens` limit you specified |
| **Natural ending** | Model generated an end-of-sequence token |
| **Stop sequence** | Encountered a predefined stop phrase |

---

## The API Response Object

The response sent back contains:

| Field | Content |
|---|---|
| `message` | The generated text |
| `usage` | Count of input tokens and output tokens |
| `stop_reason` | Why generation ended (max tokens / end token / stop sequence) |

---

## Key Takeaways

- Always route API calls through **your own server** — never the client
- Set **`max_tokens`** deliberately based on your use case
- Handle all three **stop reasons** in your application logic
- Issues can be traced to a specific phase — useful for debugging

# Making Your First Anthropic API Request (Python)

---

## 1. Environment Setup

Install required packages:
```python
%pip install anthropic python-dotenv
```

Create a `.env` file in the same directory as your notebook:
```
ANTHROPIC_API_KEY="your-api-key-here"
```

> Always add `.env` to your `.gitignore` — never commit API keys to version control.

---

## 2. Load Keys and Create Client
```python
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"
```

- `load_dotenv()` reads the `.env` file and injects the key as an environment variable
- `Anthropic()` automatically picks up `ANTHROPIC_API_KEY` from the environment

---

## 3. The `client.messages.create()` Function

The core function for all API requests. Three required parameters:

| Parameter | Type | Description |
|---|---|---|
| `model` | `str` | Which Claude model to use |
| `max_tokens` | `int` | **Safety ceiling** on response length — not a target |
| `messages` | `list` | The conversation history sent to Claude |

**Note on `max_tokens`:** Claude writes what it thinks is appropriate and stops naturally.
`max_tokens` only kicks in as a hard cutoff — it does not pad or target a length.

---

## 4. Message Structure

A conversation is a list of message dictionaries, each with two fields:
```python
{
    "role": "user",       # either "user" or "assistant"
    "content": "..."      # the actual text
}
```

| Role | Written by |
|---|---|
| `user` | You / the human |
| `assistant` | Claude's responses |

---

## 5. Making a Request
```python
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence."
        }
    ]
)
```

---

## 6. Extracting the Response
```python
message.content[0].text
```

The full response object contains metadata (token usage, stop reason, etc.),
but `.content[0].text` gives you the clean generated text directly.

---

## Full Working Example
```python
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence."
        }
    ]
)

print(message.content[0].text)
```

---

## Key Takeaways

- Never hardcode API keys — always use `.env` + `python-dotenv`
- `max_tokens` is a **safety limit**, not a length target
- The `messages` list is where conversation history lives
- Response text is at `message.content[0].text`

In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"


In [3]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

In [4]:
print(message)

print(message.content[0].text)

Message(id='msg_016uLgabtqjZ9NeZ2qseT5Ju', container=None, content=[TextBlock(citations=None, text='Quantum computing is a revolutionary computing paradigm that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that can potentially solve certain complex problems exponentially faster than classical computers.', type='text')], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=16, output_tokens=43, server_tool_use=None, service_tier='standard'))
Quantum computing is a revolutionary computing paradigm that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that can potentially solve certain complex problems exponentially faster tha

In [5]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

# Multi-Turn Conversations with the Anthropic API

---

## Key Concept: Claude is Stateless

> The Anthropic API and Claude **do not store any messages** between requests.
> Every request is completely independent — Claude has no memory of previous exchanges.

---

## The Problem This Creates
```
You:   "What is quantum computing?"
Claude: [gives a good answer]

You:   "Write another sentence"
Claude: [writes a sentence about something random — has no context]
```

Without conversation history, follow-up questions break completely.

---

## The Solution: Manual Conversation State

To have a real multi-turn conversation you must:

1. **Manually maintain a list of all messages** in your code
2. **Send the complete message history** with every follow-up request

### How the Flow Works
```
1. Send initial user message
         |
         v
2. Get Claude's response → append to messages list as "assistant"
         |
         v
3. Add follow-up question → append to messages list as "user"
         |
         v
4. Send entire messages list with next request
         |
         v
5. Repeat
```

---

## Helper Functions

Three small functions to manage conversation state cleanly:
```python
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text
```

---

## Full Working Example
```python
# Start with an empty message list
messages = []

# Turn 1
add_user_message(messages, "Define quantum computing in one sentence")
answer = chat(messages)
add_assistant_message(messages, answer)   # <-- critical step

# Turn 2
add_user_message(messages, "Write another sentence")
final_answer = chat(messages)             # Claude now has full context
```

After turn 2, the `messages` list looks like this:
```python
[
    {"role": "user",      "content": "Define quantum computing in one sentence"},
    {"role": "assistant", "content": "<Claude's first answer>"},
    {"role": "user",      "content": "Write another sentence"}
]
```

---

## Key Takeaways

| Rule | Detail |
|---|---|
| Claude has no memory | Every API call starts fresh |
| You own the history | Maintain the `messages` list yourself |
| Always append responses | Add Claude's reply as `"assistant"` before the next turn |
| Send full history | Pass the entire list on every subsequent request |

In [10]:
# Start with an empty message list
chain_messages = []

# Add the initial user question
add_user_message(chain_messages, "Define quantum computing in one sentence")
print(chain_messages)
print()
# Get Claude's response
answer = chat(chain_messages)

# Add Claude's response to the conversation history
add_assistant_message(chain_messages, answer)
print(chain_messages)
print()


# Add a follow-up question
add_user_message(chain_messages, "Write another sentence")
print(chain_messages)
print()

# Get the follow-up response with full context
final_answer = chat(chain_messages)
print(final_answer)

[{'role': 'user', 'content': 'Define quantum computing in one sentence'}]

[{'role': 'user', 'content': 'Define quantum computing in one sentence'}, {'role': 'assistant', 'content': 'Quantum computing is a revolutionary computing paradigm that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can potentially solve certain complex problems exponentially faster than classical computers.'}]

[{'role': 'user', 'content': 'Define quantum computing in one sentence'}, {'role': 'assistant', 'content': 'Quantum computing is a revolutionary computing paradigm that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can potentially solve certain complex problems exponentially faster than classical computers.'}, {'role': 'user', 'content': 'Write another sentence'}]

Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bit

# System Prompts

---

## What is a System Prompt?

A system prompt is a plain string passed to Claude **before the conversation begins**.
It defines Claude's role, tone, and behaviour — shaping how it responds to all user messages.

---

## Why They Matter

Without a system prompt, Claude gives generic answers.  
With one, Claude behaves like a specialist in a specific role.

**Example — Math Tutor chatbot:**

| You want Claude to | You do NOT want Claude to |
|---|---|
| Give hints rather than direct answers | Immediately give the full solution |
| Walk students through problems step by step | Tell students to just use a calculator |
| Show similar examples to guide thinking | |

---

## How System Prompts Work
```python
system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

client.messages.create(
    model=model,
    messages=messages,
    max_tokens=1000,
    system=system_prompt
)
```

- Passed via the `system` parameter in `client.messages.create()`
- Claude will try to respond the way someone in that specified role would respond
- Helps keep Claude focused and on task throughout the conversation

---

## Seeing the Difference

**Student asks:** "How do I solve 5x + 2 = 3 for x?"

**Without system prompt:**
> "Subtract 2 from both sides to get 5x = 1, then divide by 5 to get x = 0.2"
> *(Full answer given immediately)*

**With math tutor system prompt:**
> "What do you think would be a good first step to isolate x?
> Consider what operation we might need to perform on both sides..."
> *(Guides the student to think it through)*

---

## Building a Flexible `chat()` Function

The Anthropic API does **not** accept `system=None` — so conditionally include it:
```python
def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text
```

### Usage
```python
# Without system prompt
answer = chat(messages)

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)
```

---

## Key Takeaways

- System prompts transform generic AI responses into **role-appropriate interactions**
- They are passed once and apply to the **entire conversation**
- Do not pass `system=None` — conditionally include the parameter only when it has a value
- Essential for building consistent, purposeful AI applications

In [12]:

system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

client.messages.create(
    model=model,
    messages=chain_messages,
    max_tokens=1000,
    system=system_prompt
)

Message(id='msg_01Hfwt2rCMms7brBEf7qKuF8', container=None, content=[TextBlock(citations=None, text="I notice we've moved away from math - are you working on any specific math problems or concepts you'd like to explore together? I'm here to help guide you through mathematical thinking step by step!", type='text')], model='claude-sonnet-4-20250514', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=92, output_tokens=43, server_tool_use=None, service_tier='standard'))

In [13]:
def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
chain_messages = [{"role":"user", "content": "what is 5x+2=3, solve for x"}]

# Without system prompt
answer = chat(chain_messages)
print("Without System Prompt")
print("*"*120)
print(answer)
print("*"*120)

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(chain_messages, system=system)
print("With System Prompt")
print("*"*120)
print(answer)
print("*"*120)


Without System Prompt
************************************************************************************************************************
To solve 5x + 2 = 3 for x, I'll isolate x by performing the same operations on both sides of the equation.

Starting equation: 5x + 2 = 3

Step 1: Subtract 2 from both sides
5x + 2 - 2 = 3 - 2
5x = 1

Step 2: Divide both sides by 5
5x ÷ 5 = 1 ÷ 5
x = 1/5

Therefore, x = 1/5 or 0.2

To verify: 5(1/5) + 2 = 1 + 2 = 3 ✓
************************************************************************************************************************
With System Prompt
************************************************************************************************************************
I'll help you solve this equation step by step! Let's work through it together.

You have the equation: 5x + 2 = 3

The goal is to isolate x (get x by itself on one side). 

First, let's think about what operations we see in this equation. What is being done to x on the left s

# Temperature

---

## What is Temperature?

Temperature is a decimal value between **0.0 and 1.0** that controls how predictable
or creative Claude's responses are — essentially a "creativity dial".

---

## How Claude Generates Text (Quick Recap)

Every token Claude produces goes through three steps:
```
1. Tokenization  →  break input into tokens
2. Prediction    →  calculate probability for each possible next token
3. Sampling      →  select a token based on those probabilities
```

Temperature influences **step 3** — how that selection is made.

---

## How Temperature Affects Token Selection

| Setting | Effect | Token probabilities |
|---|---|---|
| **Low (near 0)** | Almost always picks the highest probability token | `about: 100%, rest: ~0%` |
| **High (near 1)** | Spreads probability more evenly across candidates | `about: 23%, would: 18%, of: 15%...` |
```
Low temperature  →  deterministic, consistent, predictable
High temperature →  random, varied, creative
```

---

## Temperature Ranges and Use Cases
```
Less creative <-----------------------------------------> More creative
     0.0                   0.5                              1.0
```

| Range | Label | Best for |
|---|---|---|
| **0.0 – 0.3** | Low | Factual responses, coding assistance, data extraction, content moderation |
| **0.4 – 0.7** | Medium | Summarization, educational content, problem-solving, creative writing with constraints |
| **0.8 – 1.0** | High | Brainstorming, creative writing, marketing content, joke generation |

---

## Implementing Temperature in Code

Add `temperature` as a parameter to your `chat()` function:
```python
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature     # <-- added
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text
```

### Usage
```python
# Consistent, factual output
answer = chat(messages, temperature=0.0)

# Creative, varied output
answer = chat(messages, temperature=1.0)
```

---

## Key Takeaways

- Temperature does **not guarantee** different outputs — it changes the **probability** of them
- Even at high temperature, Claude may occasionally produce similar responses
- Match temperature to your task:

| Need | Temperature |
|---|---|
| Consistent, factual answers | Low (0.0 – 0.3) |
| General purpose / balanced | Medium (0.4 – 0.7) |
| Creative brainstorming | High (0.8 – 1.0) |

# Streaming Responses

---

## The Problem with Standard (Non-Streaming) Responses
```
User sends message
       |
       v
Server waits for Claude to finish generating (10–30 seconds)
       |
       v
Full response sent to client all at once
```

Users stare at a loading spinner with no feedback — poor experience.

---

## How Streaming Fixes This

With streaming enabled, Claude sends back the response **token by token** as it generates,
and your server forwards each chunk to the client immediately.
```
User sends message
       |
       v
Claude sends initial response (acknowledges request)
       |
       v
"Quantum"  → event → server → client (text appears)
"computing" → event → server → client (text grows)
"is..."     → event → server → client (text grows)
       |
       v
All events are part of a single request
```

---

## Stream Event Types

Each streaming response is made up of a sequence of typed events:

| Event | Purpose |
|---|---|
| `MessageStart` | A new message is being sent |
| `ContentBlockStart` | Start of a new block (text, tool use, or other) |
| **`ContentBlockDelta`** | **Chunk of actual generated text** ← this is what you display |
| `ContentBlockStop` | Current content block has been completed |
| `MessageDelta` | Current message is complete |
| `MessageStop` | End of information about the current message |

### Event sequence for a single response:
```
MessageStart
    └── ContentBlockStart
            └── ContentBlockDelta  ← "Quantum"
            └── ContentBlockDelta  ← "computing"
            └── ContentBlockDelta  ← "is..."
        ContentBlockStop
    MessageDelta
MessageStop
```

---

## Implementation

### Basic Streaming (raw events)
```python
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True            # <-- enable streaming
)

for event in stream:
    print(event)           # prints every raw event object
```

---

### Simplified Text Streaming (recommended)

Use the SDK's `.stream()` context manager to get just the text — no manual event parsing:
```python
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")    # prints each chunk as it arrives
```

- `stream.text_stream` automatically filters out everything except `ContentBlockDelta` text
- `end=""` prevents newlines between chunks so text flows continuously

---

### Getting the Complete Message After Streaming

You often need the full assembled response for storage or further processing:
```python
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        pass                              # forward each chunk to client

    final_message = stream.get_final_message()  # full message object
```

- Streaming gives users real-time feedback
- `get_final_message()` gives your backend the complete response for storage/logic

---

## Key Takeaways

- Add `stream=True` to `messages.create()` for raw event access
- Use `client.messages.stream()` + `stream.text_stream` for clean text-only streaming
- `ContentBlockDelta` events carry the actual generated text
- Use `stream.get_final_message()` after the loop when you need the complete response

In [ ]:
#Basic Streaming Implementation
#To enable streaming, add stream=True to your messages.create call:

messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01K1By9esTsH7m3KrevmkE3R', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='The', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' "PetersonCorp Customer Analytics Database" is a comprehensive MySQL system', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' containing fabricated records o

In [23]:
# Simplified Text Streaming
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

FakeDataGen is a synthetic database containing 10 million artificially generated customer records with realistic but fictitious names, addresses, purchase histories, and demographic information designed for software testing and development purposes.

In [22]:
#Getting the Complete Message

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()

# Controlling Output Format: Prefilling + Stop Sequences

---

## The Problem

When asking Claude to generate structured data, it defaults to being "helpful" by adding
explanatory text and markdown formatting around the content:
````
```json
{
  "source": ["aws.ec2"],
  ...
}
```
This rule captures EC2 instance state changes...
````

For applications that need **raw, clean output** (JSON to copy, code to execute, etc.),
this extra wrapping creates friction.

> The goal: get **only the content we asked for and nothing else.**

---

## The Solution: Assistant Prefilling + Stop Sequences

Two techniques combined:

| Technique | What it does |
|---|---|
| **Assistant prefilling** | Pre-populate the start of Claude's reply to steer its output |
| **Stop sequences** | Tell Claude to stop generating when it hits a specific string |

---

## How It Works — Step by Step
````python
messages = []

add_user_message(messages, "Generate a very short EventBridge rule as JSON")
add_assistant_message(messages, "```json")   # <-- prefill: Claude thinks it already started

text = chat(messages, stop_sequences=["```"])  # <-- stop when Claude tries to close the block
````

### Claude's internal reasoning:
````
1. "OK, I need to write a full rule and describe it"
2. "Oh wait — I've already started the ```json block"  ← prefill effect
3. [writes only the JSON body]
4. "All done, I'll close it with ```"  ← STOP — generation ends here
````

### Result — clean JSON, nothing else:
````json
{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}
````

---

## Cleaning Up the Response

The response may have leading/trailing whitespace or newlines — strip and parse:
````python
import json

clean_json = json.loads(text.strip())
````

---

## Updating `chat()` to Support Stop Sequences
````python
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text
````

---

## Beyond JSON — General Pattern

This technique works for any structured output:

| Use case | Prefill with | Stop sequence |
|---|---|---|
| JSON data | ` ```json ` | ` ``` ` |
| Python code | ` ```python ` | ` ``` ` |
| Bulleted list | `- ` | *(empty line or custom)* |
| CSV data | first header row | newline / custom marker |

**Rule of thumb:** identify what Claude naturally wraps your content in →
use that as your prefill start and stop sequence end.

---

## Key Takeaways

- Assistant prefilling makes Claude believe it has already begun a specific format
- Stop sequences halt generation the moment a closing marker appears
- Together they produce **clean, parseable output** with no extra commentary
- Always `.strip()` the response before parsing structured data

In [26]:
messages = []

def chat(messages, system=None, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
print(text)


{
  "Name": "OrderProcessingRule",
  "EventPattern": {
    "source": ["myapp.orders"],
    "detail-type": ["Order Placed"]
  },
  "Targets": [
    {
      "Id": "1",
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"
    }
  ],
  "State": "ENABLED"
}



In [27]:
import json

# Clean up and parse the JSON
clean_json = json.loads(text.strip())
clean_json

{'Name': 'OrderProcessingRule',
 'EventPattern': {'source': ['myapp.orders'], 'detail-type': ['Order Placed']},
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}],
 'State': 'ENABLED'}